# Feature Engineering Combination v4 — RGB CNN Baru + AVG/NTSC Lama

Menggantikan **hanya blok RGB** pada fitur v2. Fitur AVG, NTSC, HOG, LBP, HSV, dan statistik RGB diambil dari arsip v2 yang telah divalidasi. Membuat kandidat A (RGB original + AVG/NTSC original) dan B (RGB flip-average + AVG/NTSC flip-average). Tidak ada training CNN dan tidak memuat official test. Seluruh output v1/v2/v3 dipertahankan.

## 1. Konfigurasi lokasi artefak

Gunakan `RGB Extraction v4` dan arsip Feature Engineering Combination v2 sebagai sumber. Output baru hanya di direktori `cifar10_7fitur_v4` sehingga tidak terjadi overwrite terhadap eksperimen terdahulu.

In [1]:
from pathlib import Path
import hashlib,json,os
import numpy as np
SEED=30092026
RUN_ID='cifar10_7fitur_v4'
RGB_RUN='cifar10_rgb_residual_v4'
ROOT=Path.cwd().resolve()
if not (ROOT/'outputs'/'cifar10_7fitur_v2').is_dir() and (ROOT/'tugas-7'/'outputs'/'cifar10_7fitur_v2').is_dir():ROOT=ROOT/'tugas-7'
V2=ROOT/'outputs'/'cifar10_7fitur_v2'/'fusion'
RGB_ROOT=ROOT/'outputs'/RGB_RUN/'rgb'
V4=ROOT/'outputs'/RUN_ID/'fusion'
for folder in ('fitur','laporan','model'):(V4/folder).mkdir(parents=True,exist_ok=True)
BLOCK_NAMES=['rgb','avg','ntsc','hog','lbp','hsv','rgb_stats']
BOUNDARIES=[0,512,1024,1536,1860,2116,2164,2173]
def sha256_file(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for c in iter(lambda:f.read(1024*1024),b''):h.update(c)
    return h.hexdigest()
print('Output:',V4)

Output: /home/saga/anaconda3/envs/envGPU/BDA_Project/tugas-after-uts/outputs/cifar10_7fitur_v4/fusion


## 2. Validasi manifest v2 dan model-model beku

Model AVG dan NTSC harus identik dengan yang direkam di arsip v2. RGB memakai model baru; arsitektur `svm_features` dan hash model harus sesuai metadata extraction v4.

In [2]:
manifest_v2_path=V2/'laporan'/'combination_manifest.json'
manifest_v2=json.loads(manifest_v2_path.read_text(encoding='utf-8'))
if manifest_v2['seed']!=SEED or manifest_v2['run_id']!='cifar10_7fitur_v2':raise ValueError('Manifest v2 salah')
if manifest_v2['block_names']!=BLOCK_NAMES:raise ValueError('Urutan kelompok v2 berubah')
old_model_hashes=manifest_v2['model_hashes']
for i,domain in ((1,'avg'),(2,'ntsc')):
    p=ROOT/'outputs'/'cifar10_7fitur_v1'/domain/'model'/'cnn.keras'
    if sha256_file(p)!=old_model_hashes[i]:raise ValueError(f'Checkpoint {domain} berubah')
new_model_path=RGB_ROOT/'model'/'cnn.keras'
new_meta=json.loads((RGB_ROOT/'laporan'/'cnn_training.json').read_text(encoding='utf-8'))
if sha256_file(new_model_path)!=new_meta['cnn_sha256']:raise ValueError('Checkpoint RGB v4 berubah')
model_hashes=[new_meta['cnn_sha256'],old_model_hashes[1],old_model_hashes[2]]
print('Checkpoint RGB baru, AVG/NTSC lama terverifikasi')

Checkpoint RGB baru, AVG/NTSC lama terverifikasi


## 3. Validasi fitur RGB v4 dan kedua arsip v2

Kecocokan label, indices, split, dimensi dan finiteness diperiksa sebelum penggabungan. Arsip v2 harus mengikuti `combination_manifest.json` agar descriptor dari sampel yang sama digunakan.

In [3]:
rgb_path=RGB_ROOT/'fitur'/'train_features.npz'
with np.load(rgb_path,allow_pickle=False) as a:
    if (str(a['cnn_sha256'])!=model_hashes[0] or str(a['split_hash'])!=manifest_v2['split_hash']
        or str(a['run_id'])!=RGB_RUN or str(a['layer_name'])!='svm_features'):
        raise ValueError('Provenance RGB v4 tidak sesuai')
    rgb_A=a['features_A'].astype(np.float32)
    rgb_B=a['features_B'].astype(np.float32)
    y=a['y'].astype(np.int64)
    indices=a['indices'].astype(np.int64)
    idx_train=a['train_idx'].astype(np.int64)
    idx_val=a['val_idx'].astype(np.int64)
if rgb_A.shape!=(50000,512) or rgb_B.shape!=(50000,512):raise ValueError('Dimensi RGB v4 salah')
if not np.isfinite(rgb_A).all() or not np.isfinite(rgb_B).all():raise ValueError('RGB v4 tidak berhingga')
if not np.array_equal(indices,np.arange(50000)) or len(idx_train)!=40000 or len(idx_val)!=10000:raise ValueError('Indeks RGB salah')
old={}
for rep in ('A','B'):
    p=V2/'fitur'/f'train_combined_{rep}.npz'
    if sha256_file(p)!=manifest_v2['archive_sha256'][rep]:raise ValueError(f'Hash arsip v2 {rep} salah')
    with np.load(p,allow_pickle=False) as a:
        if (a['features'].shape!=(50000,2173) or str(a['split_hash'])!=manifest_v2['split_hash']
            or not np.array_equal(a['y'],y) or not np.array_equal(a['indices'],indices)
            or not np.array_equal(a['train_idx'],idx_train) or not np.array_equal(a['val_idx'],idx_val)
            or not np.array_equal(a['boundaries'],BOUNDARIES)
            or not np.array_equal(a['model_hashes'],old_model_hashes)):
            raise ValueError(f'Alignment v2 {rep} salah')
        old[rep]=a['features'].astype(np.float32)
print('Valid:',rgb_A.shape,rgb_B.shape,'| Split:',manifest_v2['split_hash'][:16])

Valid: (50000, 512) (50000, 512) | Split: 16004a1ac8a89ac9


## 4. Kombinasi fitur A dan B

Hanya 512 kolom RGB yang diganti. `old[rep][:,512:]` berisi AVG, NTSC, HOG, LBP, HSV dan statistik RGB dengan urutan v2 yang sama. Bobot feature group akan dipelajari dalam notebook Training, bukan di sini.

In [4]:
combined={}
for rep,new_rgb in (('A',rgb_A),('B',rgb_B)):
    combined[rep]=np.concatenate((new_rgb,old[rep][:,512:]),axis=1).astype(np.float32)
    if combined[rep].shape!=(50000,2173) or not np.isfinite(combined[rep]).all():
        raise ValueError(f'Matriks v4 {rep} tidak valid')
    assert np.array_equal(combined[rep][:,512:],old[rep][:,512:])
    print(f'Kandidat {rep}:',combined[rep].shape)

Kandidat A: (50000, 2173)
Kandidat B: (50000, 2173)


## 5. Simpan kandidat terpisah dengan pemeriksaan ulang

Arsip A/B menyimpan provenance RGB v4, sumber v2, hash checkpoint ketiga CNN, indeks training-validation, dan blok tujuh fitur. Jika ada file v4 yang sudah tersimpan, cocokkan hash sebelum menerima agar percobaan tidak tercampur.

In [5]:
archives={}
for rep,X in combined.items():
    target=V4/'fitur'/f'train_combined_{rep}.npz'
    if target.exists():
        with np.load(target,allow_pickle=False) as old_file:
            if (str(old_file['run_id'])!=RUN_ID or not np.array_equal(old_file['model_hashes'],model_hashes)
                or str(old_file['split_hash'])!=manifest_v2['split_hash']
                or old_file['features'].shape!=X.shape
                or not np.array_equal(old_file['features'],X)):
                raise ValueError(f'Arsip v4 {rep} sudah ada dengan isi berbeda')
    else:
        tmp=target.with_suffix('.npz.tmp')
        with open(tmp,'wb') as stream:
            np.savez_compressed(stream,features=X,y=y,indices=indices,train_idx=idx_train,val_idx=idx_val,
                run_id=RUN_ID,representation=rep,source_run='v2+new_RGB_v4',
                split_hash=manifest_v2['split_hash'],preprocessing_version='v4-rgb-pooling-replacement',
                block_names=np.array(BLOCK_NAMES),boundaries=np.array(BOUNDARIES,dtype=np.int32),
                model_hashes=np.array(model_hashes))
        os.replace(tmp,target)
    archives[rep]=sha256_file(target)
    print('Arsip:',target.name,'| SHA:',archives[rep][:16])

Arsip: train_combined_A.npz | SHA: b52ff1dfa11c3e36
Arsip: train_combined_B.npz | SHA: af8584951025d488


## 6. Kunci manifest kombinasi v4

Manifest digunakan Training dan Testing untuk memeriksa kesamaan input dan checkpoint. Tidak ada modifikasi terhadap v2 ataupun model AVG/NTSC.

In [6]:
manifest={'run_id':RUN_ID,'seed':SEED,'split_hash':manifest_v2['split_hash'],
          'model_hashes':model_hashes,'archive_sha256':archives,'block_names':BLOCK_NAMES,
          'boundaries':{'A':BOUNDARIES,'B':BOUNDARIES},
          'rgb_model_path':str(new_model_path.relative_to(ROOT)),
          'avg_model_path':'outputs/cifar10_7fitur_v1/avg/model/cnn.keras',
          'ntsc_model_path':'outputs/cifar10_7fitur_v1/ntsc/model/cnn.keras',
          'rgb_archive_sha256':sha256_file(rgb_path),'v2_manifest_sha256':sha256_file(manifest_v2_path),
          'rgb_feature_layer':'svm_features','gray_feature_layer':'embedding',
          'preprocessing_version':'v4-rgb-pooling-replacement',
          'representation_A':'original','representation_B':'original_horizontal_flip_average'}
path=V4/'laporan'/'combination_manifest.json'
if path.exists():
    if json.loads(path.read_text(encoding='utf-8'))!=manifest:
        raise ValueError('Manifest v4 yang sudah ada berbeda; jangan menimpanya')
else:
    tmp=path.with_suffix('.json.tmp');tmp.write_text(json.dumps(manifest,indent=2,ensure_ascii=False),encoding='utf-8');os.replace(tmp,path)
print('Manifest siap:',path)

Manifest siap: /home/saga/anaconda3/envs/envGPU/BDA_Project/tugas-after-uts/outputs/cifar10_7fitur_v4/fusion/laporan/combination_manifest.json
